# Pokemon Type Classification | DL Assignment 1

## Task 2 — CNN

**Goal:** classify 3600 Pokémon images into 9 types using CNN with multiple different architectures. The performance will be compared against the MLP from Task 1.

**Metric:** Macro-averaged F1 — used because class imbalance is 2.76× (Water 674 vs Ground 244). Accuracy would be misleading.

_____________
**DO NOT RUN ALL CELLS**

> This notebook requires some 20 minutes to run all training sequences with T4 GPU. So if you don't have those resources, avoid running it all again and **instead just run the needed specific cells you want.**
______

---
## Table of Contents

- [Part 0 — Project Load](#part-0---project-load)
  - [Config](#config)
  - [Import & Set Up](#import--set-up)
  - [Load Data](#load-data)
- [Part 1 — Exploratory Data Analysis](#part-1--exploratory-data-analysis)
  - [Plot 1 — Class Distribution](#plot-1--class-distribution)
  - [Plot 2 — Sample Images per Class](#plot-2--sample-images-per-class)
  - [Plot 3 — Average Image per Class](#plot-3--average-image-per-class)
  - [Plot 4 — Per-Channel Pixel Statistics](#plot-4--per-channel-pixel-statistics)
  - [Plot 5 — Pixel Intensity Histogram](#plot-5--pixel-intensity-histogram)
  - [Plot 6 — PCA → t-SNE Cluster Plot](#plot-6--pca--t-sne-cluster-plot)
- [Part 2 — Model Experiments](#part-2--model-experiments)
  - [Results Manager](#results-manager)
  - [Setup Experiments](#setup-experiments)
  - [Experiments Plan & Registry](#experiments-plan--registry)
  - [A: Vanilla Baseline](#a-vanilla-baseline)
  - [B: MLP Baseline](#b-mlp-baseline)
  - [C: Dropout(0.3) + label_smoothing=0.1](#c-dropout03--label_smoothing01)
  - [D: + weight_decay=1e-4](#d--weight_decay1e-4)
  - [E: WeightedRandomSampler](#e-weightedrandomsampler)
  - [F: Narrow + Deep MLP](#f-narrow--deep-mlp)
  - [G: Bottleneck MLP](#g-bottleneck-mlp)
  - [H: VanillaMLP_v2 (wider, no regularisation)](#h-vanillamlp_v2-wider-no-regularisation)
  - [I: VanillaMLP_v2 + targeted class weights](#i-vanillamlp_v2--targeted-class-weights-rock--ground)
  - [J: MLP with lighter dropout (0.2)](#j-mlp-with-lighter-dropout-02)
  - [K: VanillaMLP_v2 + tiny L2](#k-vanillamlp_v2--tiny-l2-weight_decay1e-5)
  - [L: WiderMLP (1024 first layer) + winning C recipe](#l-widermlp-1024-first-layer--winning-c-recipe)
  - [M: C architecture + WeightedRandomSampler](#m-c-architecture--weightedrandomsampler-no-loss-weights)
  - [N: C architecture + CosineAnnealingLR](#n-c-architecture--cosineannealinglr)
  - [O: C + WeightedRandomSampler + class weights (both)](#o-c--weightedrandomsampler--class-weights-both-imbalance-corrections)
  - [P: MLP with very light dropout (0.15)](#p-mlp-with-very-light-dropout-015--winning-c-recipe)
  - [Q: WiderMLP + WeightedRandomSampler](#q-widermlp--weightedrandomsampler-no-class-weights-in-loss)
  - [R: C architecture + stronger label smoothing (LS=0.15)](#r-c-architecture--stronger-label-smoothing-ls015)
  - [S: DeepMLP (4-layer funnel)](#s-deepmlp-4-layer-funnel-512256128640--winning-c-recipe)
- [Part 3 — Pre-Processing & Data Augmentation](#part-3--pre-processing--data-augmentation)
  - [3.1 — Pre-Processing: Grayscale](#31--pre-processing-grayscale)
  - [3.2 — Data Augmentation Exploration](#32--data-augmentation-exploration)
- [Part 4 — Comparison, Final Model & Evaluation](#part-4--comparison-final-model--evaluation)
  - [4.1: Soft Ensemble Exploration](#41-soft-ensemble-exploration)
  - [4.2: All Models Summary](#42-all-models-summary)
  - [4.3: Best model deep dive](#43-best-model-deep-dive)
- [Final Summary & Submission](#final-summary--submission)

---

**About the code:**

> This notebook is the runner + readable story.

> This notebook clones the repo from github, downloads the data from google drive, runs the code cells, during which saves results to the outputs folder and at the end does the automatic download of that folder.

> All reusable logic lives in `src/`.
> - `src/models/mlp.py` — 7 MLP classes: MLP, VanillaMLP, VanillaMLP_v2, NarrowMLP, WiderMLP, BottleneckMLP, DeepMLP — all with `in_channels` param (RGB=3 or grayscale=1)
> - `src/datasets/dataset.py` — RGB, augmented, grayscale, and gray+equalize transform pipelines; `grayscale`/`equalize` params in loaders
> - `src/training/early_stopping.py` — metric-agnostic, monitors by minimisation (`stopper(-val_f1, model)`)
> - `src/evaluation/metrics.py`, `plots.py`, `submission.py` — classification report, training curves, confusion matrix, submission CSV
> - `src/evaluation/ensemble.py` — soft ensemble with `inference_mode` for test-set inference (test labels are uuid strings, not ints)
> - `src/evaluation/persistence.py` — `save_outputs`/`restore_outputs` (Drive backup), `download_and_extract` (dataset download), `restore_tracker` (load prior results)
> - Tests: `models_test.py` (14), `dataset_test.py` (14), `train_test.py` (3) — all pass locally


**AI Assistance Disclosure**

> Artificial intelligence tools (Claude, ChatGPT, GitHub Copilot) were used strategically to enhance development efficiency while maintaining full academic integrity and authorship responsibility. AI assistance was limited to:

> - Debugging & testing — writing unit tests, identifying bugs and validating expected behaviours across the codebase

> - Visualisation & reporting — implementing plots and formatting results for clarity and readability, and adding comments to document the code

> All core concepts, research methodology, architectural decisions, and algorithmic implementations were conceived and developed by the authors.

# Part 0 - Project Load

### Config

In [2]:
# All run-level constants live HERE.
# Change FAST_RUN to switch between a quick smoke-test and a real Colab run.
# Nothing else needs touching.


# ==============  FLIP THESE FLAGS  ===============================
FAST_RUN         = False   # True = smoke-test (tiny data, 2 epochs) | False = real Colab run
SAVE_IN_EACH_RUN = False   # True = zip outputs to DRIVE_OUTPUTS_DIR after every experiment
USE_DRIVE        = True   # True = enable save/restore of outputs zip (Drive on Colab, local folder elsewhere)
# =================================================================

# === Task identifier — change this ONE constant when copying the notebook for Task 2 / Task 3.
TASK_NAME = "task2"

# ==== Drive / backup settings =====================================
# Shared Drive folder that holds the dataset zip and output backups.
# Anyone-with-the-link public share URL — no hardcoded personal MyDrive paths.
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1u2Xw2-4_L5OhPFjY_OsgNbMayrTI-ROR?usp=sharing"

# Name of the dataset zip inside DRIVE_FOLDER_URL.
DATA_FILE_NAME = "the-pokemon-are-out-there-2026-task-1.zip"

# Name of the outputs zip inside DRIVE_FOLDER_URL.
OUTPUTS_FILE_NAME = TASK_NAME + "_outputs.zip"

# Where to save/restore the outputs zip:
#   Colab → inside your personal MyDrive (auto-mounted)
#   Local → any local folder (created automatically if needed)
import sys as _sys
DRIVE_OUTPUTS_DIR = (
    "/content/drive/MyDrive/DL_Proj"
    if "google.colab" in _sys.modules
    else str(__import__("pathlib").Path.home() / "DL_Proj")
)


# ======== Training hyperparams =============================================
EPOCHS      = 2     if FAST_RUN else 30
# PATIENCE: val_macro_f1 is noisier than val_loss (small val set → 1 mis-prediction
# swings F1 by ~1.2%). Use PATIENCE=7 to avoid stopping too early on noisy F1 plateaus.
PATIENCE    = 1     if FAST_RUN else 7
LR          = 1e-3
BATCH_SIZE  = 32    if FAST_RUN else 64
IMG_SIZE    = 64    # MLP input: 64x64x3 = 12288 flat features
NUM_WORKERS = 8

# smoke-test data size: 9 classes × N = total training images
N_SAMPLES_PER_CLASS = 6   # 54 total — enough to run the full pipeline in seconds

print(f"TASK_NAME         : {TASK_NAME}")
print(f"FAST_RUN          : {FAST_RUN}  (N_SAMPLES_PER_CLASS={N_SAMPLES_PER_CLASS if FAST_RUN else 'all'})")
print(f"EPOCHS            : {EPOCHS}  |  PATIENCE : {PATIENCE}  |  LR : {LR}")
print(f"BATCH_SIZE        : {BATCH_SIZE}  |  IMG_SIZE : {IMG_SIZE}x{IMG_SIZE}")
print(f"USE_DRIVE         : {USE_DRIVE}  |  SAVE_IN_EACH_RUN : {SAVE_IN_EACH_RUN}")
print(f"DRIVE_FOLDER_URL  : {DRIVE_FOLDER_URL}")
print(f"DRIVE_OUTPUTS_DIR : {DRIVE_OUTPUTS_DIR}")


TASK_NAME         : task2
FAST_RUN          : False  (N_SAMPLES_PER_CLASS=all)
EPOCHS            : 30  |  PATIENCE : 7  |  LR : 0.001
BATCH_SIZE        : 64  |  IMG_SIZE : 64x64
USE_DRIVE         : True  |  SAVE_IN_EACH_RUN : False
DRIVE_FOLDER_URL  : https://drive.google.com/drive/folders/1u2Xw2-4_L5OhPFjY_OsgNbMayrTI-ROR?usp=sharing
DRIVE_OUTPUTS_DIR : C:\Users\18ilv\DL_Proj


### Import & Set Up

> Includes the autoreload, to automatically reload the src modules before every cell run, so if we edit some src/ file, we don't need to restart the kernel and loose all our state.

> This works locally. If we are running the notebook in colab, and we change the src files locally, so we need to push those changes to GitHub and then run the cell bellow for colab to go and get those new src/ files.

In [ ]:
# ── PROJECT IMPORT AND SET UP ─────────────────────────────────────────────────────────

import sys, os, time, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if not IN_COLAB:
  %load_ext autoreload
  %autoreload 2

if IN_COLAB:
    if not os.path.exists("/content/DL_Proj"):
        !git clone https://github.com/fmssilva/DL_Proj.git /content/DL_Proj
    else:
        !git -C /content/DL_Proj pull --ff-only
    os.chdir("/content/DL_Proj/assignment_1")
    %pip install -r requirements.txt -q
else:
    # notebook kernel starts in task1/ — step up to assignment_1/ root
    cwd = Path(os.getcwd())
    if cwd.name.startswith("task"):
        os.chdir(cwd.parent)

ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
import pandas as pd
import matplotlib.pyplot as plt

from src.config import (
    SEED, CLASSES, NUM_CLASSES, DATA_DIR, OUT_DIR,
    get_task_out_dir, set_seed,
)

set_seed(SEED)

TASK_OUT_DIR = get_task_out_dir(TASK_NAME)
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CSV_PATH     = DATA_DIR / "train_labels.csv"
TRAIN_DIR    = DATA_DIR / "Train"
TEST_DIR     = DATA_DIR / "Test"

# auto-detect number of CPU cores for DataLoader workers (safe on Colab and local)
import multiprocessing
NUM_WORKERS = min(NUM_WORKERS, multiprocessing.cpu_count())

print("-"*50)
print(f"ROOT        : {ROOT}")
print(f"TASK_NAME   : {TASK_NAME}  |  TASK_OUT_DIR : {TASK_OUT_DIR}")
print(f"EPOCHS      : {EPOCHS}  |  PATIENCE : {PATIENCE}  |  LR : {LR}")
print(f"BATCH_SIZE  : {BATCH_SIZE}  |  IMG_SIZE : {IMG_SIZE}x{IMG_SIZE}")
print(f"NUM_WORKERS : {NUM_WORKERS}")
print(f"Device      : {device}  |  PyTorch {torch.__version__}")
print(f"Outputs     : {TASK_OUT_DIR.resolve()}")


### Hot Reload

> If we did chnges in src/ files and pushed to GitHub, now we just need to run this cell for colab to pull the new files.

In [ ]:
import importlib
import sys
from pathlib import Path

if IN_COLAB:
    # pull latest code from GitHub before reloading
    import os
    os.chdir(Path(ROOT).parent)
    os.system("git pull --ff-only")
    os.chdir(ROOT)

# reload every src.* module that is already loaded in this kernel
_reloaded, _skipped = [], []
for mod_name, mod in list(sys.modules.items()):
    if mod_name.startswith("src.") and hasattr(mod, "__file__") and mod.__file__:
        try:
            importlib.reload(mod)
            _reloaded.append(mod_name)
        except Exception as e:
            print(f"  [WARN] could not reload {mod_name}: {e}")
            _skipped.append(mod_name)

print(f"Reloaded {len(_reloaded)} src modules: {_reloaded}")
if _skipped:
    print(f"Skipped  {len(_skipped)}: {_skipped}")


### Load Data

In [ ]:
from src.evaluation.persistence import download_and_extract

# Download and extract the dataset if not already present.
# Uses DRIVE_FOLDER_URL + DATA_FILE_NAME from Config — works on Colab and locally.
if not Path("data/train_labels.csv").exists():
    ok = download_and_extract(DRIVE_FOLDER_URL, DATA_FILE_NAME, "data/")
    if not ok:
        print(
            "ERROR: data download failed.\n"
            f"  Folder : {DRIVE_FOLDER_URL}\n"
            f"  File   : {DATA_FILE_NAME}\n"
            "  Check that the folder is publicly shared and the file name is correct."
        )
else:
    print("data/ already present — skipping download")


df_full = pd.read_csv(CSV_PATH)
print(f"Full dataset : {len(df_full)} rows, {df_full['label'].nunique()} classes")

# FAST_RUN: subsample N per class so every cell finishes fast on CPU
if FAST_RUN:
    df = df_full.groupby("label", group_keys=False).head(N_SAMPLES_PER_CLASS).reset_index(drop=True)
    print(f"FAST_RUN     : subsampled to {len(df)} rows ({N_SAMPLES_PER_CLASS}/class)")
else:
    df = df_full

print(f"\nClass counts used this run:\n{df['label'].value_counts().to_string()}")

# Part 1 — Model Experiments

## Results Manager
> This global dictionary keeps track of the models we train along the notebook, so it is easy to keep track and compare all models we train.

> This global dictionary with models results can be initialized empty, or we can choose to upload some previous output zip folder that we have, so we update this global results manager dictionary with all the models we got in previous runs.

**Run this cell only if you want to upload the saved task_outputs.zip from google drive and restore the results_tracker dictionary with that data.**

In [ ]:
from src.evaluation.persistence import restore_outputs

# Restore previous outputs from Drive/local backup.
# On Colab  : mounts Drive and reads from DRIVE_OUTPUTS_DIR (private mounted path).
# Locally   : downloads from DRIVE_FOLDER_URL (public shared folder).
# Skip entirely when USE_DRIVE=False — nothing to restore.

if USE_DRIVE:
    backup_dir = DRIVE_OUTPUTS_DIR if IN_COLAB else DRIVE_FOLDER_URL
    restore_outputs(TASK_OUT_DIR, TASK_NAME, in_colab=IN_COLAB, backup_dir=backup_dir)
else:
    print("[restore] USE_DRIVE=False — skipping restore, starting fresh.")

In [ ]:
from src.evaluation.persistence import restore_tracker, ResultsTracker

# ResultsTracker = dict[str, ExperimentEntry]
# Each entry has these keys (see ExperimentEntry in persistence.py):
#   val_macro_f1 : float  — best val macro-F1 from the saved checkpoint
#   val_acc      : float  — val accuracy at that checkpoint
#   val_loss     : float  — val loss at that checkpoint (nan for ensembles)
#   total_epochs : int    — epochs actually run (< EPOCHS if early-stopped; 0 for ensembles)
#   train_time_s : float  — wall-clock training time in seconds (0.0 for ensembles)
#   history      : dict   — {"train_loss": [...], "val_loss": [...], "train_f1": [...], "val_f1": [...]}
#   child_models : list   — only present on ensemble entries; names of the averaged solo models
results_tracker: ResultsTracker = {}

results_json_path = TASK_OUT_DIR / "results" / f"{TASK_NAME}_results.json"


def _print_leaderboard(tracker: ResultsTracker) -> None:
    """Show all experiments sorted by val_macro_f1 descending as a styled DataFrame."""
    if not tracker:
        return
    rows = sorted(tracker.items(), key=lambda x: x[1]["val_macro_f1"], reverse=True)
    data = [
        {
            "rank":    rank,
            "name":    name,
            "val_F1":  round(m["val_macro_f1"], 4),
            "val_acc": round(m["val_acc"], 4),
            "epochs":  m["total_epochs"],
            "time(s)": round(m["train_time_s"], 1),
        }
        for rank, (name, m) in enumerate(rows, 1)
    ]
    df_lb = pd.DataFrame(data).set_index("rank")
    try:
        from IPython.display import display
        styled = (
            df_lb.style
            .background_gradient(subset=["val_F1"], cmap="RdYlGn", vmin=0.10, vmax=df_lb["val_F1"].max())
            .background_gradient(subset=["val_acc"], cmap="Blues", vmin=0.10, vmax=df_lb["val_acc"].max())
            .format({"val_F1": "{:.4f}", "val_acc": "{:.4f}", "time(s)": "{:.1f}"})
        )
        display(styled)
    except Exception:
        # terminal / non-Jupyter fallback
        print(df_lb.to_string())


print("-" * 50)
restore_tracker(results_json_path, results_tracker)
print("Current experiments in results_tracker:")
_print_leaderboard(results_tracker)


## Setup Experiments
These helper cells define the shared infrastructure used by every experiment:
- **`build_loaders(augment, use_sampler, grayscale, equalize)`** — wraps `get_train_val_loaders`, always uses `df` (which can be the subsampled FAST_RUN set)
- **`run_experiment(model, name, criterion, optimizer, scheduler, loaders)`** — full train loop, early stopping on `val_macro_f1`, checkpoint save, evaluate best, store in `results_tracker` and immediately upsert to `task2_results.json`

Each experiment cell is then just 4–5 lines: instantiate model, criterion, optimizer, scheduler → call `run_experiment`.

In [ ]:

# ── Shared imports for all experiment cells ───────────────────────────────────
import torch.nn as nn
from torch.utils.data import DataLoader
from src.datasets.dataset import (
    PokemonDataset, compute_class_weights,
    get_base_transforms, get_gray_transforms, get_train_val_loaders,
)
from src.models.cnn import CNN, LeNet5
from src.training.train import train_one_epoch, evaluate
from src.training.early_stopping import EarlyStopping
from src.evaluation.metrics import classification_report_str
from src.evaluation.plots import plot_history, plot_confusion_matrix
from src.evaluation.persistence import save_experiment_result, save_all_results, save_outputs
from src.evaluation.submission import generate_submission_from_preds, validate_submission

import time

# build class weights from the run's df (correct proportions even for FAST_RUN)
label_to_idx     = {cls: i for i, cls in enumerate(CLASSES)}
_all_train_labels = [label_to_idx[lbl] for lbl in df["label"]]
class_weights    = compute_class_weights(_all_train_labels).to(device)

# test loader is always the full test set — not touched until submission
test_ds     = PokemonDataset(TEST_DIR, get_base_transforms(IMG_SIZE), csv_path=None)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# single path for incremental per-experiment saves and final save_all_results
RESULTS_PATH = TASK_OUT_DIR / "results" / f"{TASK_NAME}_results.json"
SUB_PATH     = TASK_OUT_DIR / "results" / f"submission_{TASK_NAME}.csv"

print(f"Test images  : {len(test_ds)}")
print(f"Class weights: { {cls: round(class_weights[i].item(), 3) for i, cls in enumerate(CLASSES)} }")


# ── Helper: build_loaders ─────────────────────────────────────────────────────
def build_loaders(augment: bool = False, use_sampler: bool = False,
                  grayscale: bool = False, equalize: bool = False):
    """Return (train_loader, val_loader) for the current df and hyperparams.
    grayscale=True → (1,H,W) tensors, input_dim=4096. equalize=True → histogram stretch before gray."""
    return get_train_val_loaders(
        CSV_PATH, TRAIN_DIR, IMG_SIZE, BATCH_SIZE,
        augment=augment, use_sampler=use_sampler,
        num_workers=NUM_WORKERS, df_override=df,
        grayscale=grayscale, equalize=equalize,
    )


# pre-build the two standard loader pairs used by all Part 2 experiments
loaders_base    = build_loaders(augment=False, use_sampler=False)
loaders_sampler = build_loaders(augment=False, use_sampler=True)


# ── Helper: _make_test_loader ─────────────────────────────────────────────────
def _make_test_loader(stem: str) -> DataLoader:
    """Return a test DataLoader with the correct transform for the given experiment stem.
    Gray experiments need grayscale transforms; all others use the standard RGB transform."""
    if stem.endswith("_gray_eq"):
        t = get_gray_transforms(IMG_SIZE, equalize=True)
    elif stem.endswith("_gray"):
        t = get_gray_transforms(IMG_SIZE, equalize=False)
    else:
        return test_loader   # reuse the global RGB test loader
    ds = PokemonDataset(TEST_DIR, t, csv_path=None)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


# ── Helper: _is_new_overall_best ─────────────────────────────────────────────
def _is_new_overall_best(name: str, f1: float) -> bool:
    """True if this experiment (solo or ensemble) beats every other entry in results_tracker.
    Compares against all existing entries so the CSV is only updated when we truly have a new top."""
    other_f1s = [v["val_macro_f1"] for k, v in results_tracker.items() if k != name]
    return f1 > max(other_f1s, default=-1)


# ── Helper: run_experiment ────────────────────────────────────────────────────
def run_experiment(model, name: str, criterion, optimizer, scheduler, loaders) -> tuple:
    """
    Train model up to EPOCHS with early stopping on val macro-F1.
    Saves best checkpoint to {TASK_OUT_DIR}/checkpoints/<name>.pth.
    Upserts this experiment into {TASK_NAME}_results.json after each run.
    If this experiment is the new overall best (beats solo models AND ensembles),
    auto-regenerates submission_{TASK_NAME}.csv.
    Returns (model_with_best_weights, history_dict).
    """
    train_loader, val_loader = loaders
    ckpt_path = str(TASK_OUT_DIR / "checkpoints" / f"{name}.pth")
    stopper   = EarlyStopping(patience=PATIENCE, checkpoint_path=ckpt_path)
    history   = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []}
    t0        = time.time()

    print(f"\n{'='*60}")
    print(f"  Experiment: {name}")
    print(f"  Model params: {sum(p.numel() for p in model.parameters()):,}")
    print(f"{'='*60}")

    for epoch in range(1, EPOCHS + 1):
        ep_t0         = time.time()
        train_loss    = train_one_epoch(model, train_loader, criterion, optimizer, device)
        train_metrics = evaluate(model, train_loader, criterion, device)
        val_metrics   = evaluate(model, val_loader,   criterion, device)
        elapsed       = time.time() - ep_t0
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["train_f1"].append(train_metrics["macro_f1"])
        history["val_f1"].append(val_metrics["macro_f1"])

        print(
            f"  Epoch {epoch:02d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f}  train_f1={train_metrics['macro_f1']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f}  val_f1={val_metrics['macro_f1']:.4f} | "
            f"{elapsed:.1f}s"
        )

        # negate val_f1 because EarlyStopping minimises — this saves the peak-F1 checkpoint
        stopper(-val_metrics["macro_f1"], model)
        if stopper.stop:
            print(f"  Early stopping at epoch {epoch} (patience={PATIENCE}, metric: val_macro_f1).")
            break

    total_time = time.time() - t0

    # reload best checkpoint and run a final clean evaluation
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    best = evaluate(model, val_loader, criterion, device)

    entry = {
        "val_macro_f1": round(best["macro_f1"], 4),
        "val_acc":      round(best["acc"], 4),
        "val_loss":     round(best["loss"], 4),
        "total_epochs": len(history["train_loss"]),
        "train_time_s": round(total_time, 1),
        "history":      history,
    }
    results_tracker[name] = entry

    print(f"\n  Best checkpoint: val_loss={best['loss']:.4f}  val_macro_f1={best['macro_f1']:.4f}")
    print(f"  Total time: {total_time:.1f}s ({total_time/len(history['train_loss']):.1f}s/epoch)")
    _print_leaderboard(results_tracker)

    # save this experiment's result to disk immediately — no data lost if Colab disconnects
    save_experiment_result(name, entry, RESULTS_PATH)

    # auto-update submission if this solo model beats everyone (solo + ensemble)
    if _is_new_overall_best(name, entry["val_macro_f1"]):
        print(f"\n  New overall best — regenerating submission CSV...")
        _tl = _make_test_loader(name)
        test_preds = []
        model.eval()
        with torch.no_grad():
            for imgs, _ in _tl:
                imgs = imgs.to(device)
                test_preds.extend(model(imgs).argmax(dim=1).cpu().tolist())
        generate_submission_from_preds(_tl, test_preds, CLASSES, SUB_PATH)
        validate_submission(SUB_PATH)
        print(f"  Submission updated -> {SUB_PATH}")

    if SAVE_IN_EACH_RUN:
        save_outputs(TASK_OUT_DIR, TASK_NAME, in_colab=IN_COLAB, use_drive=USE_DRIVE, drive_dir=DRIVE_OUTPUTS_DIR)

    return model, history


## Experiments Plan & Registry

In [ ]:
# Registry: one entry per experiment stem.
# "model"      — zero-arg factory for the RGB version (in_channels=3).
# "model_gray" — zero-arg factory for the grayscale version (in_channels=1).
# "criterion"  — zero-arg factory; must match what was used during training.
#
# Grayscale (_gray, _gray_eq) and augmented (_augmented) variants are registered
# dynamically in Part 3 cells — their stems depend on best_name at runtime.
model_registry = {
    # ── RGB experiments ────────────────────────────────────────────────────────
    "A_cnn": {
        "model": lambda: CNN(img_size=IMG_SIZE),
        "model_gray": lambda: CNN(img_size=IMG_SIZE, in_channels=1),
        "criterion": lambda: nn.CrossEntropyLoss()},
}


### A: CNN
###### ──────────────────────────────────────────

In [ ]:

# This instantiates our model and moves it to device (gpu/cpu)
model_A = CNN(img_size=IMG_SIZE).to(device)

# This defines the loss functin
criterion_A = nn.CrossEntropyLoss()   # no class weights — true bare baseline

# This defines the optimizer for gradient descent
optimizer_A = torch.optim.Adam(model_A.parameters(), lr=LR)

# This defines a learning rate scheduler to dynamically adjust the LR during training
scheduler_A = torch.optim.lr_scheduler.StepLR(optimizer_A, step_size=5, gamma=0.5)

model_A, history_A = run_experiment(model_A, "A_CNN", criterion_A, optimizer_A, scheduler_A, loaders_base)